In [5]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

25/12/09 00:51:32 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/09 00:51:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-afce3298-f56c-44b8-9b8f-42d60a3c2003;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [6]:
# Cell 0 – Utils import (shared pattern)

import sys
import os
import importlib

# Path to the folder that contains utils_silver.py
# (adjust if your repo is in a different location)
project_root = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"
silver_dir = os.path.join(project_root, "_02_Silver")

if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

import utils_silver
importlib.reload(utils_silver)  # pick up edits during dev
from utils_silver import *

print("Imported utils_silver from:", utils_silver.__file__)


✅ utils_silver.py loaded
Imported utils_silver from: /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 — Imports + Path Setup

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

from utils_silver import build_paths, enforce_schema, dq_expect, write_metric

storage_account = "clientdatastorage"
paths_bronze, paths_silver, paths_gold = build_paths(storage_account)

BRONZE_PROVIDERS_PATH = paths_bronze["providers"]
SILVER_PROVIDERS_PATH = paths_silver["providers"]
SILVER_CLAIMS_PATH    = paths_silver["claims"]  # we’ll use Silver claims as reference

spark.sql("CREATE DATABASE IF NOT EXISTS bupa_silver")
print("Bronze providers path:", BRONZE_PROVIDERS_PATH)
print("Silver providers path:", SILVER_PROVIDERS_PATH)
print("Silver claims path   :", SILVER_CLAIMS_PATH)



Bronze providers path: abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers
Silver providers path: abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers
Silver claims path   : abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims


# Cell 2 — Read Bronze Providers

In [4]:
providers_bz = (
    spark.read
         .format("delta")
         .load(BRONZE_PROVIDERS_PATH)
)

print("Bronze providers:")
print("rows:", providers_bz.count())
providers_bz.show(5)
providers_bz.printSchema()


Bronze providers:


25/12/04 17:54:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


rows: 5410
+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|   PRV51262|         0|
|   PRV51606|         0|
|   PRV51657|         0|
|   PRV51665|         0|
|   PRV51749|         0|
+-----------+----------+
only showing top 5 rows

root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)



# Cell 3 – Read Claims (Silver) to capture all Provider_IDs actually used

In [6]:
claims_silver = (
    spark.read
         .format("delta")
         .load(SILVER_CLAIMS_PATH)
)

print("Claims rows:", claims_silver.count())
claims_silver.select("Claim_ID", "Provider_ID").show(5)


Claims rows: 558211


+---------+-----------+
| Claim_ID|Provider_ID|
+---------+-----------+
|CLM110031|       NULL|
|CLM110044|   PRV54888|
|CLM110046|   PRV53609|
|CLM110050|   PRV54690|
|CLM110055|   PRV55946|
+---------+-----------+
only showing top 5 rows



# Cell 4 – Build unified Provider dimension (superset)

In [7]:
# 1) All provider IDs referenced in claims
providers_from_claims = (
    claims_silver
        .select("Provider_ID")
        .filter(F.col("Provider_ID").isNotNull())
        .dropDuplicates()
)

print("Distinct Provider_IDs from claims:", providers_from_claims.count())

# 2) Clean bronze providers (just enforce schema)
provider_schema = StructType([
    StructField("Provider_ID", StringType(), True),
    StructField("Fraud_Flag", IntegerType(), True),
])

providers_bz_clean = enforce_schema(providers_bz, provider_schema)

# 3) LEFT JOIN → attach fraud label where available
providers_all = (
    providers_from_claims
      .join(providers_bz_clean, on="Provider_ID", how="left")
      .withColumn(
          "Fraud_Flag",
          F.coalesce(F.col("Fraud_Flag"), F.lit(0)).cast("int")
      )
      .withColumn(
          "Fraud_Label_Source",
          F.when(F.col("Fraud_Flag").isNull(), F.lit("assumed_clean"))
           .otherwise(F.lit("labelled"))
      )
)

print("Unified providers (claims-driven) rows:", providers_all.count())
providers_all.show(10, truncate=False)
providers_all.printSchema()



Distinct Provider_IDs from claims: 5393


Unified providers (claims-driven) rows: 5393


+-----------+----------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|
+-----------+----------+------------------+
|PRV54400   |0         |labelled          |
|PRV55039   |1         |labelled          |
|PRV53872   |0         |labelled          |
|PRV53223   |0         |labelled          |
|PRV54888   |0         |labelled          |
|PRV54182   |0         |labelled          |
|PRV55099   |0         |labelled          |
|PRV51262   |0         |labelled          |
|PRV51665   |0         |labelled          |
|PRV55541   |0         |labelled          |
+-----------+----------+------------------+
only showing top 10 rows

root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = false)
 |-- Fraud_Label_Source: string (nullable = false)



Business logic here:

If Kaggle had a label → we keep it

If not → we keep the provider as a row, but mark them fraud_flag=0, source="assumed_clean"

# Cell 5 – Basic DQ checks (PK, type)

In [8]:
# Provider_ID must not be null
dq_expect(
    providers_all,
    name="provider_pk_not_null",
    expr="Provider_ID IS NOT NULL",
    severity="error",
    table_label="providers",
    paths_silver=paths_silver
)

print("After DQ checks, rows:", providers_all.count())


✅ DQ PASS [providers] provider_pk_not_null


After DQ checks, rows: 5393


If there are any null Provider_IDs here, it’s a modelling issue; but in practice it should be fine because we filtered nulls in providers_from_claims.

# Cell 6 – Final Silver providers DataFrame

In [9]:
providers_silver = providers_all.select(
    "Provider_ID",
    "Fraud_Flag",
    "Fraud_Label_Source"
)

providers_silver.show(10, truncate=False)
providers_silver.printSchema()
print("Silver Providers rowcount:", providers_silver.count())


+-----------+----------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|
+-----------+----------+------------------+
|PRV54400   |0         |labelled          |
|PRV55039   |1         |labelled          |
|PRV53872   |0         |labelled          |
|PRV53223   |0         |labelled          |
|PRV54888   |0         |labelled          |
|PRV54182   |0         |labelled          |
|PRV55099   |0         |labelled          |
|PRV51262   |0         |labelled          |
|PRV51665   |0         |labelled          |
|PRV55541   |0         |labelled          |
+-----------+----------+------------------+
only showing top 10 rows

root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = false)
 |-- Fraud_Label_Source: string (nullable = false)



Silver Providers rowcount: 5393


# Cell 7 – Write to Delta path

In [10]:
print("Writing Silver providers to:", SILVER_PROVIDERS_PATH)

(providers_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PROVIDERS_PATH)
)

print("✅ Silver providers written.")


Writing Silver providers to: abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers


✅ Silver providers written.


# Cell 8 – Register as table

In [11]:
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_silver")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS bupa_silver.providers
USING DELTA
LOCATION '{SILVER_PROVIDERS_PATH}'
""")

print("✅ Registered table: bupa_silver.providers")

spark.table("bupa_silver.providers").show(10, truncate=False)



✅ Registered table: bupa_silver.providers
+-----------+----------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|
+-----------+----------+------------------+
|PRV54400   |0         |labelled          |
|PRV55039   |1         |labelled          |
|PRV53872   |0         |labelled          |
|PRV53223   |0         |labelled          |
|PRV54888   |0         |labelled          |
|PRV54182   |0         |labelled          |
|PRV55099   |0         |labelled          |
|PRV51262   |0         |labelled          |
|PRV51665   |0         |labelled          |
|PRV55541   |0         |labelled          |
+-----------+----------+------------------+
only showing top 10 rows



# Cell 9 – Metrics

In [12]:
from inspect import signature
print("write_metric signature:", signature(write_metric))

write_metric(
    spark,
    name="rowcount_providers_silver",
    value=providers_silver.count(),
    context="providers_silver",
    paths_silver=paths_silver
)

write_metric(
    spark,
    name="distinct_provider_ids_silver",
    value=providers_silver.select("Provider_ID").distinct().count(),
    context="providers_silver",
    paths_silver=paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)



write_metric signature: (spark, name: str, value, context: str, paths_silver: dict)


[METRIC] rowcount_providers_silver=5393 ctx=providers_silver


[METRIC] distinct_provider_ids_silver=5393 ctx=providers_silver


+----------------------------+------+----------------+--------------------------+
|metric                      |value |context         |ts                        |
+----------------------------+------+----------------+--------------------------+
|distinct_provider_ids_silver|5393  |providers_silver|2025-12-04 18:07:28.694462|
|rowcount_providers_silver   |5393  |providers_silver|2025-12-04 18:07:20.906509|
|distinct_provider_ids       |5410  |providers_silver|2025-11-29 00:42:09.375763|
|rowcount_providers_silver   |5410  |providers_silver|2025-11-29 00:42:04.874106|
|distinct_claim_ids          |558211|claims_silver   |2025-11-28 15:31:16.213598|
|rowcount_claims_silver      |558211|claims_silver   |2025-11-28 15:30:54.237362|
|distinct_member_ids         |381109|members_silver  |2025-11-28 14:04:23.229343|
|rowcount_members_silver     |381109|members_silver  |2025-11-28 14:04:14.937709|
|distinct_policy_ids_silver  |381109|policies_silver |2025-11-28 10:36:21.144919|
|rowcount_polici

# Cell 10 – (Optional) Quick comparison & FK sanity view

In [13]:
# Compare coverage: how many claim providers now exist in providers_silver?

claims = claims_silver  # alias

prov_silver = providers_silver.select("Provider_ID").dropDuplicates()

claims_with_flag = (
    claims
      .join(prov_silver, "Provider_ID", "left")
      .withColumn("provider_fk_known", F.when(F.col("Provider_ID").isNotNull(), 1).otherwise(0))
)

total_claims = claims_with_flag.count()
unknown_prov_claims = claims_with_flag.filter(F.col("provider_fk_known") == 0).count()

print({
    "claims_total": total_claims,
    "claims_with_known_provider_pct": round((total_claims - unknown_prov_claims) / total_claims * 100, 2),
    "claims_with_unknown_provider_pct": round(unknown_prov_claims / total_claims * 100, 2),
})



{'claims_total': 558211, 'claims_with_known_provider_pct': 92.99, 'claims_with_unknown_provider_pct': 7.01}


3. How to explain this decision in an interview

You can say something like:

“I noticed that some provider IDs in claims didn’t have corresponding rows in the labelled providers dataset.
Rather than overwriting IDs or losing claims, I expanded the Providers dimension: I took all distinct Provider_IDs from claims and left-joined them onto the labelled providers table. Where Kaggle had fraud labels, I kept them; where it didn’t, I defaulted Fraud_Flag to 0 and tagged them as ‘assumed_clean’.
This way, every claim has a valid provider reference in Providers Silver, but I still preserve the distinction between truly labelled providers and unlabeled ones.”

That shows:

you understand data integrity

you avoid “cheating” by changing keys

you design realistic, explainable pipelines

# ⭐ A–D: SILVER PROVIDERS ANALYSIS CELLS (Same Style as Policies/Members/Claims)

# A) Bronze vs Silver Comparison

In [14]:
# A) Snapshot basics: row counts & distinct Provider_IDs

from pyspark.sql import functions as F

bronze_prov = spark.read.format("delta").load(BRONZE_PROVIDERS_PATH)
silver_prov = spark.read.format("delta").load(SILVER_PROVIDERS_PATH)

metrics_prov = {
    "rows_bronze_providers": bronze_prov.count(),
    "rows_silver_providers": silver_prov.count(),
    "distinct_provider_id_bronze": bronze_prov.select("Provider_ID").distinct().count(),
    "distinct_provider_id_silver": silver_prov.select("Provider_ID").distinct().count(),
}

print("=== Providers snapshot (Bronze vs Silver) ===")
for k, v in metrics_prov.items():
    print(f"{k}: {v}")



=== Providers snapshot (Bronze vs Silver) ===
rows_bronze_providers: 5410
rows_silver_providers: 5393
distinct_provider_id_bronze: 5410
distinct_provider_id_silver: 5393


This will show you clearly that Silver has at least all providers used in claims, and potentially more than Bronze (because we expanded using claims).

# 🔹 B) Label coverage (labelled vs assumed-clean)B) Per-Column Change Report

In [15]:
# B) Label coverage in Silver providers

from pyspark.sql import functions as F

silver_prov = spark.read.format("delta").load(SILVER_PROVIDERS_PATH)

print("Silver Providers schema:")
silver_prov.printSchema()

if "Fraud_Label_Source" in silver_prov.columns:
    # Group by label source
    label_stats = (
        silver_prov
            .groupBy("Fraud_Label_Source")
            .agg(
                F.count("*").alias("rows"),
                F.countDistinct("Provider_ID").alias("distinct_providers"),
                F.sum("Fraud_Flag").alias("sum_fraud_flag")
            )
    )
    print("=== Fraud label coverage by source ===")
    label_stats.show(truncate=False)

    # Percentage by source
    label_counts = {
        r["Fraud_Label_Source"]: r["distinct_providers"]
        for r in label_stats.collect()
    }
    total_providers = sum(label_counts.values()) or 1
    pct = {k: round(v / total_providers * 100.0, 2) for k, v in label_counts.items()}

    print("Provider label coverage (% of providers by source):")
    for source, p in pct.items():
        print(f"  {source}: {p}%")

else:
    print("⚠️ Column 'Fraud_Label_Source' not found. Showing Fraud_Flag distribution only:")
    silver_prov.groupBy("Fraud_Flag").count().orderBy("Fraud_Flag").show()



Silver Providers schema:
root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)
 |-- Fraud_Label_Source: string (nullable = true)

=== Fraud label coverage by source ===
+------------------+----+------------------+--------------+
|Fraud_Label_Source|rows|distinct_providers|sum_fraud_flag|
+------------------+----+------------------+--------------+
|labelled          |5393|5393              |506           |
+------------------+----+------------------+--------------+

Provider label coverage (% of providers by source):
  labelled: 100.0%


This tells a great story in interviews:

“labelled” = providers that had Kaggle fraud labels

“assumed_clean” = providers that only appear in claims, so we default Fraud_Flag = 0

# 🔹 C) FK coverage: Claims → Providers

In [16]:
# C) FK coverage: how well claims match providers_silver

from pyspark.sql import functions as F

claims_silver = spark.read.format("delta").load(SILVER_CLAIMS_PATH)
silver_prov   = spark.read.format("delta").load(SILVER_PROVIDERS_PATH)

# 1) Basic counts
total_claims = claims_silver.count()
claims_with_provider_id = claims_silver.filter(F.col("Provider_ID").isNotNull()).count()

# 2) Check how many of those Provider_IDs are known in providers_silver
prov_keys = silver_prov.select(F.col("Provider_ID").alias("Prov_Key")).dropDuplicates()

claims_fk = (
    claims_silver
        .withColumn("has_provider_id", F.when(F.col("Provider_ID").isNull(), 0).otherwise(1))
        .join(prov_keys, claims_silver["Provider_ID"] == prov_keys["Prov_Key"], "left")
)

claims_with_known_provider = claims_fk.filter(
    (F.col("has_provider_id") == 1) & F.col("Prov_Key").isNotNull()
).count()

claims_with_unknown_provider = claims_fk.filter(
    (F.col("has_provider_id") == 1) & F.col("Prov_Key").isNull()
).count()

summary_fk = {
    "claims_total": total_claims,
    "claims_with_provider_id": claims_with_provider_id,
    "claims_with_known_provider": claims_with_known_provider,
    "claims_with_unknown_provider": claims_with_unknown_provider,
    "pct_with_known_provider": round(
        claims_with_known_provider / max(claims_with_provider_id, 1) * 100.0, 4
    ),
    "pct_with_unknown_provider": round(
        claims_with_unknown_provider / max(claims_with_provider_id, 1) * 100.0, 4
    ),
}

print("=== Claims → Providers FK coverage (Silver) ===")
for k, v in summary_fk.items():
    print(f"{k}: {v}")



=== Claims → Providers FK coverage (Silver) ===
claims_total: 558211
claims_with_provider_id: 519069
claims_with_known_provider: 519069
claims_with_unknown_provider: 0
pct_with_known_provider: 100.0
pct_with_unknown_provider: 0.0


In an ideal world with our “superset providers” logic,
pct_with_known_provider should be ≈ 100% (apart from rows where Provider_ID itself is null).

# 🔹 D) Feature coverage & distributions in Providers Silver

In [17]:
# D) Feature coverage & distributions in Providers Silver

from pyspark.sql import functions as F

silver_prov = spark.read.format("delta").load(SILVER_PROVIDERS_PATH)

expected_cols = ["Provider_ID", "Fraud_Flag", "Fraud_Label_Source"]
exists = {c: (c in silver_prov.columns) for c in expected_cols}

print("=== Providers Silver feature existence ===")
for c, ok in exists.items():
    print(f"{c}: {ok}")

print("\nFraud_Flag distribution:")
silver_prov.groupBy("Fraud_Flag").count().orderBy("Fraud_Flag").show()

if "Fraud_Label_Source" in silver_prov.columns:
    print("Fraud_Label_Source distribution:")
    silver_prov.groupBy("Fraud_Label_Source").count().orderBy("Fraud_Label_Source").show()


=== Providers Silver feature existence ===
Provider_ID: True
Fraud_Flag: True
Fraud_Label_Source: True

Fraud_Flag distribution:
+----------+-----+
|Fraud_Flag|count|
+----------+-----+
|         0| 4887|
|         1|  506|
+----------+-----+

Fraud_Label_Source distribution:
+------------------+-----+
|Fraud_Label_Source|count|
+------------------+-----+
|          labelled| 5393|
+------------------+-----+



This cell helps you talk about:

how many providers are flagged as fraud (1 vs 0)

how many are labelled vs assumed clean

confirming the Silver schema matches your design